In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("batch_id", "batch_002")

batch_id = dbutils.widgets.get("batch_id")

print("Processing batch:", batch_id)

In [0]:
customers = (
    spark.table("bank_catalog.silver.customers")
    .filter(F.col("batch_id") == batch_id)
)

accounts = (
    spark.table("bank_catalog.silver.accounts")
    .filter(F.col("batch_id") == batch_id)
)

transactions = (
    spark.table("bank_catalog.silver.transactions")
    .filter(F.col("batch_id") == batch_id)
)

loans = (
    spark.table("bank_catalog.silver.loans")
    .filter(F.col("batch_id") == batch_id)
)

branches = (
    spark.table("bank_catalog.silver.branches")
    .filter(F.col("batch_id") == batch_id)
)

In [0]:
def incremental_merge(df, target_table, key):

    if spark.catalog.tableExists(target_table):

        delta_table = DeltaTable.forName(
            spark, target_table
        )

        (
            delta_table.alias("target")
            .merge(
                df.alias("source"),
                f"target.{key} = source.{key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

    print(f"Gold incremental load completed: {target_table}")

In [0]:
dim_customer = customers.select(
    "customer_id",
    "customer_code",
    "first_name",
    "last_name",
    "date_of_birth",
    "city",
    "branch_id",
    "customer_segment",
    "onboarding_date",
    "customer_status"
).dropDuplicates(["customer_id"])

incremental_merge(
    dim_customer,
    "bank_catalog.gold.dim_customer",
    "customer_id"
)

In [0]:
dim_account = accounts.select(
    "account_id",
    "customer_id",
    "branch_id",
    "account_number",
    "account_type",
    "open_date",
    "account_status",
    "currency"
).dropDuplicates(["account_id"])

incremental_merge(
    dim_account,
    "bank_catalog.gold.dim_account",
    "account_id"
)

In [0]:
dim_branch = branches.select(
    "branch_id",
    "branch_name",
    "city",
    "state",
    "branch_type",
    "branch_status"
).dropDuplicates(["branch_id"])

incremental_merge(
    dim_branch,
    "bank_catalog.gold.dim_branch",
    "branch_id"
)

In [0]:
date_source = (
    transactions
    .select(F.to_date("transaction_timestamp").alias("full_date"))
    .union(
        loans.select(
            F.col("disbursement_date").alias("full_date")
        )
    )
    .filter(F.col("full_date").isNotNull())
    .distinct()
)

dim_date = (
    date_source
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("year", F.year("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
)

incremental_merge(
    dim_date,
    "bank_catalog.gold.dim_date",
    "date_key"
)

In [0]:
accounts_all = spark.table(
    "bank_catalog.silver.accounts"
)

fact_transaction = (
    transactions.alias("t")
    .join(
        accounts_all.alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "inner"
    )
    .select(
        F.col("t.transaction_id"),
        F.col("t.account_id"),
        F.col("a.customer_id"),
        F.col("a.branch_id"),
        F.date_format(
            F.col("t.transaction_timestamp"), "yyyyMMdd"
        ).cast("int").alias("date_key"),
        F.col("t.transaction_timestamp"),
        F.col("t.transaction_type"),
        F.col("t.amount"),
        F.col("t.currency"),
        F.col("t.channel"),
        F.col("t.transaction_status"),
        F.col("t.batch_id"),
        F.lit(1).alias("transaction_count")
    )
)

incremental_merge(
    fact_transaction,
    "bank_catalog.gold.fact_transaction",
    "transaction_id"
)


In [0]:
customers_all = spark.table(
    "bank_catalog.silver.customers"
)

fact_loan = (
    loans.alias("l")
    .join(
        customers_all.alias("c"),
        F.col("l.customer_id") == F.col("c.customer_id"),
        "inner"
    )
    .select(
        F.col("l.loan_id"),
        F.col("l.customer_id"),
        F.col("l.branch_id"),
        F.date_format(
            F.col("l.disbursement_date"), "yyyyMMdd"
        ).cast("int").alias("date_key"),
        F.col("l.loan_type"),
        F.col("l.sanctioned_amount"),
        F.col("l.interest_rate"),
        F.col("l.tenure_months"),
        F.col("l.disbursement_date"),
        F.col("l.loan_status"),
        F.col("l.batch_id"),
        F.lit(1).alias("loan_count")
    )
)

incremental_merge(
    fact_loan,
    "bank_catalog.gold.fact_loan",
    "loan_id"
)

In [0]:
%sql
SELECT
    s.batch_id,
    COUNT(*) AS total_transactions
FROM bank_catalog.gold.fact_transaction g
JOIN bank_catalog.silver.transactions s
  ON g.transaction_id = s.transaction_id
GROUP BY s.batch_id
ORDER BY s.batch_id;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT transaction_id) AS unique_records
FROM bank_catalog.gold.fact_transaction;